# Creating a Simple Agent with Tracing

In [1]:
import sys
import subprocess

required = ["openai", "python-dotenv", "openai-agents"]
subprocess.check_call([sys.executable, "-m", "pip", "install", "-U", *required])

# Then your original imports:
import dotenv
import os
from openai import OpenAI
from agents import Agent, Runner, trace
from openai.types.responses import ResponseTextDeltaEvent

dotenv.load_dotenv()

if not os.environ.get("OPENAI_API_KEY"):
    print(
        """Error: OPENAI_API_KEY environment variable not set. Please copy the .env.template file as .env and fill it in.
    
    You can execute these commands in the terminal to get started:
    cp .env.template .env
    code .env
    """
    )

# Test OpenAI Access
print(
    OpenAI()
    .responses.create(
        model=os.environ["OPENAI_DEFAULT_MODEL"], input="Say: We are up and running!"
    )
    .output_text
)

We are up and running!


In [ ]:
from agents import Agent, Runner, trace
from openai.types.responses import ResponseTextDeltaEvent

Create a simple Nutrition Assistant Agent

In [4]:
nutrition_agent = Agent(
    name="Nutrition Assistant",
    instructions="""
    You are a helpful assistant for nutrition advice.
    You give concise and accurate answers to nutrition questions.
    """
)


Let's execute the Agent:

In [5]:
with trace("Simple Nutrition Agent"):
    result = await Runner.run(nutrition_agent, "How healthy are bananas?")

print(result)

RunResult:
- Last agent: Agent(name="Nutrition Assistant", ...)
- Final output (str):
    Bananas are a healthy, convenient choice as part of a balanced diet.
    
    Key nutrients per medium banana (about 118 g):
    - ~105 calories; ~27 g carbs (3 g fiber; ~14 g sugar); ~1 g protein; ~0.3 g fat
    - ~400–450 mg potassium
    - small amounts of vitamin C and vitamin B6
    
    Benefits:
    - Potassium supports heart health and blood pressure
    - Dietary fiber and resistant starch (more when unripe) aid digestion and gut health
    - Quick source of energy and helps with satiety
    
    Considerations:
    - They contain natural sugars, so portion if you’re watching sugar or calories
    - Unripe bananas have more resistant starch; ripe ones are sweeter and digested faster
    - People with kidney disease or on potassium-restricted diets should monitor intake
    
    Bottom line: nutritious, versatile, and generally beneficial when eaten as part of a varied diet.
- 2 new item(s

Streaming the answer to the screen, token by token

In [6]:
response_stream = Runner.run_streamed(nutrition_agent, "How healthy are bananas?")

async for event in response_stream.stream_events():
    if event.type == "raw_response_event" and isinstance(
        event.data, ResponseTextDeltaEvent
    ):
        print(event.data.delta, end="", flush=True)

Bananas are a healthy, convenient fruit when eaten as part of a balanced diet. Quick facts:

- Nutrients per medium banana: about 105 calories, ~27 g carbs, ~3 g fiber, ~420 mg potassium, 0.5 mg vitamin B6, ~10 mg vitamin C; negligible fat and protein.
- Benefits: good for heart health (potassium), digestion (fiber), energy (carbs) and metabolism (B6).
- Considerations: contain natural sugars; ripeness affects sugar content (unripe has more resistant starch, ripe is sweeter). GI is moderate.
- Tips: great on-the-go snack, with yogurt or peanut butter; store at room temp; refrigerating slows ripening but won’t harm the fruit.
- Special cases: if you have kidney disease or need to limit potassium, watch portions and consult a clinician.

Overall: healthy and versatile for most people.

_Good Job!_